# ANN Prediction Unseen Data

## Imports & Configuration

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import pickle
import os
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error, r2_score

SEP = os.sep


MODEL_DIR      = f'trained_mod_CBFV_ann_tf{SEP}' 
ALL_PARAM_SETS = [1]                  
N_FOLDS        = 5                                  # fold-models per param set

# Feature file — set FEAT_FILE to a pre-normalised xlsx/csv,
# OR set FEAT_FILE=None and point RAW_FEAT_FILE + xmin/xmax CSVs to scale here.
FEAT_FILE      = 'Unseen_PGM_CBFV_Ir_feats.xlsx'
RAW_FEAT_FILE  = None
XMIN_FILE      = 'xmin_PGM_CBFV_01.csv'
XMAX_FILE      = 'xmax_PGM_CBFV_01.csv'

TARGET_COL     = 'Mass_Change'
ALLOY_COL      = 'alloy_name'
OUTPUT_FILE    = 'ann_predictions.xlsx'
FEAT_COLS      = [
    'avg_number_of_valence_electrons',
    'EN_Pauling', 'r_asym', 'H_chem', 'H_el',
    'Time', 'Coh_E'
]

print('Libraries imported.')
print(f'Model directory : {os.path.abspath(MODEL_DIR)}')
print(f'Param sets      : {ALL_PARAM_SETS}')
print(f'Feature file    : {FEAT_FILE or RAW_FEAT_FILE}')


## Load Feature File & Scale

In [ ]:
print('=' * 65)
print('LOADING FEATURES')
print('=' * 65)

if FEAT_FILE is not None:
    ext = os.path.splitext(FEAT_FILE)[1].lower()
    df_full = pd.read_excel(FEAT_FILE) if ext in ('.xlsx','.xls') else pd.read_csv(FEAT_FILE)
    print(f'Loaded normalised file : {FEAT_FILE}  shape={df_full.shape}')
    missing = [c for c in FEAT_COLS + [TARGET_COL, ALLOY_COL] if c not in df_full.columns]
    if missing:
        raise KeyError(f'Missing columns in feature file: {missing}')
    x_feats_norm        = df_full[FEAT_COLS].copy()
    y_true              = df_full[TARGET_COL].values
    alloy_names         = df_full[ALLOY_COL].values
    _already_normalised = True
    print('Scaling skipped — file already normalised.')
else:
    ext = os.path.splitext(RAW_FEAT_FILE)[1].lower()
    df_full = pd.read_excel(RAW_FEAT_FILE) if ext in ('.xlsx','.xls') else pd.read_csv(RAW_FEAT_FILE)
    print(f'Loaded raw file        : {RAW_FEAT_FILE}  shape={df_full.shape}')
    missing = [c for c in FEAT_COLS + [TARGET_COL, ALLOY_COL] if c not in df_full.columns]
    if missing:
        raise KeyError(f'Missing columns in feature file: {missing}')
    xmin = pd.read_csv(XMIN_FILE, index_col=0).squeeze('columns').reindex(FEAT_COLS)
    xmax = pd.read_csv(XMAX_FILE, index_col=0).squeeze('columns').reindex(FEAT_COLS)
    miss_sc = xmin[xmin.isna()].index.tolist()
    if miss_sc:
        raise KeyError(f'Features missing from xmin/xmax CSVs: {miss_sc}')
    denom        = (xmax - xmin).replace(0, np.nan)
    x_feats_norm = (df_full[FEAT_COLS].copy() - xmin) / denom
    y_true       = df_full[TARGET_COL].values
    alloy_names  = df_full[ALLOY_COL].values
    _already_normalised = False
    oob = ((x_feats_norm < 0) | (x_feats_norm > 1)).sum()
    oob = oob[oob > 0]
    if len(oob):
        print(f'Out-of-training-range values per feature (expected for val/test):')
        print(oob.to_string())

X_array = x_feats_norm.values
print(f'\nFeature matrix shape : {X_array.shape}')
print(f'Samples              : {len(y_true)}')
display(x_feats_norm.head())

## Predictions

In [ ]:
print('=' * 65)
print('PREDICTING WITH ALL PARAM SETS')
print('=' * 65)

all_fold_preds   = {}  
ensemble_preds   = {}  
ensemble_stds    = {} 
fold_model_names = {}  

for s in ALL_PARAM_SETS:
    pattern   = f'ann-mod-set{s:02d}-K'
    mod_files = sorted([f for f in os.listdir(MODEL_DIR) if f.startswith(pattern)])
    if len(mod_files) == 0:
        raise FileNotFoundError(
            f'No models found for param set {s} '
            f'(pattern={pattern}) in {MODEL_DIR}.\n'
            f'Run ann_GPU_Op.ipynb first to train and save models.'
        )
    fold_model_names[s] = mod_files
    preds_mat = np.zeros((len(X_array), len(mod_files)))
    for k_idx, fname in enumerate(mod_files):
        mod = pickle.load(open(os.path.join(MODEL_DIR, fname), 'rb'))
        preds_mat[:, k_idx] = mod.predict(X_array)
        print(f'  Set {s} | {fname}')
    all_fold_preds[s]   = preds_mat
    ensemble_preds[s]   = preds_mat.mean(axis=1)
    ensemble_stds[s]    = preds_mat.std(axis=1)

print(f'\nDone. {len(ALL_PARAM_SETS) * N_FOLDS} models loaded and run.')


In [ ]:
all_vals = np.concatenate([y_true] + [ensemble_preds[s] for s in ALL_PARAM_SETS])
lim_lo   = float(all_vals.min()) * 0.95
lim_hi   = float(all_vals.max()) * 1.05
cmap_ov  = cm.tab10

fig, axes = plt.subplots(1, 2, figsize=(16, 7))


ax = axes[0]
for idx, s in enumerate(ALL_PARAM_SETS):
    yp     = ensemble_preds[s]
    mae_s  = mean_absolute_error(y_true, yp)
    r2_s   = r2_score(y_true, yp)
    ax.scatter(y_true, yp,
               color=cmap_ov(idx / len(ALL_PARAM_SETS)),
               s=45, alpha=0.65, edgecolors='k', linewidths=0.3,
               label=f'Set {s}  MAE={mae_s:.4f}  R2={r2_s:.3f}')
ax.plot([lim_lo, lim_hi], [lim_lo, lim_hi], 'k--', lw=1.5, label='Ideal')
ax.set_xlim(lim_lo, lim_hi); ax.set_ylim(lim_lo, lim_hi)
ax.set_xlabel('Actual Mass Change', fontsize=13)
ax.set_ylabel('Predicted Mass Change', fontsize=13)
ax.set_title('Parity — All 5 Param Sets Overlaid', fontsize=13)
ax.legend(fontsize=8, loc='upper left')
ax.grid(True, alpha=0.3)


ax = axes[1]
for idx, s in enumerate(ALL_PARAM_SETS):
    mae_s = cmp_df.loc[s, 'MAE']
    r2_s  = cmp_df.loc[s, 'R2']
    ax.scatter(mae_s, r2_s,
               color=cmap_ov(idx / len(ALL_PARAM_SETS)),
               s=200, edgecolors='k', linewidths=0.8, zorder=5)
    ax.annotate(f'Set {s}', (mae_s, r2_s),
                textcoords='offset points', xytext=(7, 5), fontsize=11)
ax.set_xlabel('MAE  (lower = better)', fontsize=12)
ax.set_ylabel('R²  (higher = better)', fontsize=12)
ax.set_title('MAE vs R² — Param Set Trade-off', fontsize=13)
ax.grid(True, alpha=0.35)

plt.suptitle('Cross-Param-Set Comparison', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('ann_all_sets_parity_overlay.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved -> ann_all_sets_parity_overlay.png')


In [ ]:

BEST_PARAM_SET = int(cmp_df['MAE'].idxmin())



model_list    = fold_model_names[BEST_PARAM_SET]
ensemble_size = len(model_list)
y_avg         = ensemble_preds[BEST_PARAM_SET]
ensemble_std  = ensemble_stds[BEST_PARAM_SET]

df_pred_all_mods = pd.DataFrame(
    all_fold_preds[BEST_PARAM_SET],
    columns=model_list
)

residuals = y_true - y_avg
mae   = mean_absolute_error(y_true, y_avg)
rmse  = float(np.sqrt(np.mean(residuals**2)))
mape  = mean_absolute_percentage_error(y_true, y_avg) * 100
r2    = r2_score(y_true, y_avg)

print(f'Best param set (lowest test MAE) : Set {BEST_PARAM_SET}')
print(f'  MAE   : {mae:.6f}')
print(f'  RMSE  : {rmse:.6f}')
print(f'  MAPE  : {mape:.3f} %')
print(f'  R2    : {r2:.6f}')
print(f'  Models: {model_list}')


## Compile Results & Save to Excel

In [ ]:
print('=' * 65)
print(f'COMPILING RESULTS')
print('=' * 65)

df_results = pd.DataFrame({
    ALLOY_COL      : alloy_names,
    TARGET_COL     : y_true,
    'pred_avg'     : y_avg,
    'pred_std'     : ensemble_std,
    'residual'     : residuals,
    'abs_residual' : np.abs(residuals),
})
df_results = pd.concat(
    [df_results, df_pred_all_mods.reset_index(drop=True)], axis=1
)

df_results.to_excel(OUTPUT_FILE, index=False)
print(f'Saved -> {OUTPUT_FILE}  shape={df_results.shape}')
display(df_results.head())


## Peannormance Metrics

In [ ]:
print('=' * 65)
print(f' PEannORMANCE METRICS ')
print('=' * 65)

metrics_df = pd.DataFrame({
    'Metric': ['MAE', 'RMSE', 'MAPE (%)', 'R2'],
    'Value' : [round(mae, 6), round(rmse, 6), round(mape, 4), round(r2, 6)]
})
display(metrics_df)

print(f'\nResidual statistics:')
print(f'  Mean : {residuals.mean():.6f}')
print(f'  Std  : {residuals.std():.6f}')
print(f'  Min  : {residuals.min():.6f}')
print(f'  Max  : {residuals.max():.6f}')


## Parity Plots Predictions

In [ ]:
lim_lo = min(float(y_true.min()), float(y_avg.min())) * 0.95
lim_hi = max(float(y_true.max()), float(y_avg.max())) * 1.05

fig, axes = plt.subplots(1, 2, figsize=(14, 6))


ax = axes[0]
ax.scatter(y_true, y_avg, c='green', s=55, alpha=0.7,
           edgecolors='k', linewidths=0.4)
ax.plot([lim_lo, lim_hi], [lim_lo, lim_hi], 'k--', lw=1.5)
ax.set_xlim(lim_lo, lim_hi); ax.set_ylim(lim_lo, lim_hi)
ax.set_xlabel('Actual Mass Change', fontsize=13, fontweight='bold')
ax.set_ylabel('Predicted Mass Change', fontsize=13, fontweight='bold')
ax.set_title(f'Parity Plot — Set {BEST_PARAM_SET}  (R2={r2:.3f}, MAE={mae:.4f})', fontsize=13)
ax.text(0.05, 0.93,
        f'MAE  = {mae:.4f}\nRMSE = {rmse:.4f}\nMAPE = {mape:.2f}%\nR2   = {r2:.4f}',
        transform=ax.transAxes, fontsize=9, verticalalignment='top',
        bbox=dict(boxstyle='round', facecolor='white', alpha=0.75))
ax.grid(True, alpha=0.3)


ax = axes[1]
ax.errorbar(y_true, y_avg, yerr=ensemble_std,
            fmt='o', color='green', alpha=0.6, markersize=6,
            ecolor='grey', elinewidth=1, capsize=3)
ax.plot([lim_lo, lim_hi], [lim_lo, lim_hi], 'k--', lw=1.5)
ax.set_xlim(lim_lo, lim_hi); ax.set_ylim(lim_lo, lim_hi)
ax.set_xlabel('Actual Mass Change', fontsize=13, fontweight='bold')
ax.set_ylabel('Predicted Mass Change', fontsize=13, fontweight='bold')
ax.set_title(f'Parity + Ensemble Uncertainty — Set {BEST_PARAM_SET}', fontsize=13)
ax.grid(True, alpha=0.3)

plt.suptitle(f'ann Ensemble — Parity Plots  (Best Set {BEST_PARAM_SET})',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('ann_parity.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved -> ann_parity.png')


## Comprehensive Analysis Dashboard 

In [ ]:
fig = plt.figure(figsize=(18, 12))

# 1 — parity
ax1 = fig.add_subplot(2, 3, 1)
ax1.scatter(y_true, y_avg, c='green', s=50, alpha=0.65, edgecolors='k', linewidths=0.4)
ax1.plot([lim_lo, lim_hi], [lim_lo, lim_hi], 'k--', lw=1.5)
ax1.set_xlim(lim_lo, lim_hi); ax1.set_ylim(lim_lo, lim_hi)
ax1.set_xlabel('Actual', fontsize=12); ax1.set_ylabel('Predicted', fontsize=12)
ax1.set_title(f'Parity  (R2={r2:.3f}, MAE={mae:.3f})', fontsize=12)
ax1.grid(True, alpha=0.3)

# 2 — parity + uncertainty
ax2 = fig.add_subplot(2, 3, 2)
ax2.errorbar(y_true, y_avg, yerr=ensemble_std,
             fmt='o', color='green', alpha=0.6, markersize=5,
             ecolor='grey', elinewidth=0.8, capsize=2)
ax2.plot([lim_lo, lim_hi], [lim_lo, lim_hi], 'k--', lw=1.5)
ax2.set_xlim(lim_lo, lim_hi); ax2.set_ylim(lim_lo, lim_hi)
ax2.set_xlabel('Actual', fontsize=12); ax2.set_ylabel('Predicted', fontsize=12)
ax2.set_title('Parity + Uncertainty', fontsize=12)
ax2.grid(True, alpha=0.3)

# 3 — residual vs predicted
ax3 = fig.add_subplot(2, 3, 3)
ax3.scatter(y_avg, residuals, c='royalblue', s=50, alpha=0.65,
            edgecolors='k', linewidths=0.4)
ax3.axhline(0, color='red', linestyle='--', lw=1.8)
ax3.set_xlabel('Predicted Mass Change', fontsize=12)
ax3.set_ylabel('Residual (Actual - Pred)', fontsize=12)
ax3.set_title('Residual Plot', fontsize=12)
ax3.grid(True, alpha=0.3)

# 4 — residual distribution
ax4 = fig.add_subplot(2, 3, 4)
ax4.hist(residuals, bins=30, color='skyblue', edgecolor='k', alpha=0.75)
ax4.axvline(0, color='red', linestyle='--', lw=1.8, label='Zero')
ax4.axvline(float(residuals.mean()), color='green', linestyle='--',
            lw=1.8, label=f'Mean {residuals.mean():.3f}')
ax4.set_xlabel('Residual', fontsize=12)
ax4.set_ylabel('Frequency', fontsize=12)
ax4.set_title('Residual Distribution', fontsize=12)
ax4.legend(fontsize=9); ax4.grid(True, alpha=0.3)

# 5 — uncertainty vs prediction (coloured by |residual|)
ax5 = fig.add_subplot(2, 3, 5)
sc = ax5.scatter(y_avg, ensemble_std, c=np.abs(residuals),
                 cmap='RdYlGn_r', s=55, alpha=0.75,
                 edgecolors='k', linewidths=0.3)
plt.colorbar(sc, ax=ax5, label='|Residual|')
ax5.set_xlabel('Predicted Mass Change', fontsize=12)
ax5.set_ylabel('Ensemble Std (uncertainty)', fontsize=12)
ax5.set_title('Prediction vs Uncertainty', fontsize=12)
ax5.grid(True, alpha=0.3)

# 6 — sample-by-sample with uncertainty band
ax6 = fig.add_subplot(2, 3, 6)
idx = np.arange(len(y_true))
ax6.scatter(idx, y_true,  c='red',   marker='x', s=90,  lw=2,  label='Actual',    zorder=5)
ax6.scatter(idx, y_avg,   c='green', s=35, alpha=0.75,         label='Predicted',  zorder=4)
ax6.fill_between(idx, y_avg - ensemble_std, y_avg + ensemble_std,
                 color='green', alpha=0.18, label='±1 Std')
ax6.set_xlabel('Sample Index', fontsize=12)
ax6.set_ylabel('Mass Change', fontsize=12)
ax6.set_title('Actual vs Predicted (with Uncertainty)', fontsize=12)
ax6.legend(fontsize=9); ax6.grid(True, alpha=0.3)

plt.suptitle(f'ann Prediction Dashboard — Best Param Set {BEST_PARAM_SET}',
             fontsize=15, fontweight='bold', y=1.005)
plt.tight_layout()
plt.savefig('ann_dashboard.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved -> ann_dashboard.png')


## Per-Fold Comparison

In [ ]:
print('=' * 65)
print(f'PER-FOLD METRICS')
print('=' * 65)

ind_rows = []
for mod_name in model_list:
    yp = df_pred_all_mods[mod_name].values
    ind_rows.append({
        'Model'  : mod_name,
        'MAE'    : round(mean_absolute_error(y_true, yp), 5),
        'RMSE'   : round(float(np.sqrt(np.mean((y_true - yp)**2))), 5),
        'MAPE_%' : round(mean_absolute_percentage_error(y_true, yp) * 100, 3),
        'R2'     : round(r2_score(y_true, yp), 4)
    })
ind_rows.append({'Model': '>>> Ensemble Avg',
                 'MAE': round(mae, 5), 'RMSE': round(rmse, 5),
                 'MAPE_%': round(mape, 3), 'R2': round(r2, 4)})

ind_df = pd.DataFrame(ind_rows)
display(ind_df)

# overlay parity for all folds
fig, ax = plt.subplots(figsize=(9, 7))
cmap_f = plt.cm.tab10
for i, mod_name in enumerate(model_list):
    ax.scatter(y_true, df_pred_all_mods[mod_name],
               alpha=0.35, s=30, color=cmap_f(i / ensemble_size),
               label=f'Fold {i+1}')
ax.scatter(y_true, y_avg, c='red', s=70, marker='x',
           linewidths=2, label='Ensemble Avg', zorder=10)
ax.plot([lim_lo, lim_hi], [lim_lo, lim_hi], 'k--', lw=1.5, label='Ideal')
ax.set_xlim(lim_lo, lim_hi); ax.set_ylim(lim_lo, lim_hi)
ax.set_xlabel('Actual Mass Change', fontsize=13)
ax.set_ylabel('Predicted Mass Change', fontsize=13)
ax.set_title(f'Individual Fold Predictions vs Ensemble — Set {BEST_PARAM_SET}', fontsize=13)
ax.legend(fontsize=9, ncol=2); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('ann_individual_models.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved -> ann_individual_models.png')


## Outlier & High-Uncertainty Export


In [ ]:
print('=' * 65)
print(f'OUTLIER ANALYSIS ')
print('=' * 65)

# high uncertainty
unc_thresh = float(ensemble_std.mean()) + 2 * float(ensemble_std.std())
high_unc   = df_results[df_results['pred_std'] > unc_thresh]
print(f'High-uncertainty threshold : {unc_thresh:.4f}')
print(f'High-uncertainty samples   : {len(high_unc)}')
if len(high_unc):
    display(high_unc[[ALLOY_COL, TARGET_COL, 'pred_avg', 'pred_std']]
            .sort_values('pred_std', ascending=False).head(10))
    high_unc.to_excel('ann_high_uncertainty_samples.xlsx', index=False)
    print('Saved -> ann_high_uncertainty_samples.xlsx')

print()

# large errors
err_thresh = float(np.abs(residuals).mean()) + 2 * float(np.abs(residuals).std())
large_err  = df_results[df_results['abs_residual'] > err_thresh]
print(f'Large-error threshold      : {err_thresh:.4f}')
print(f'Large-error samples        : {len(large_err)}')
if len(large_err):
    display(large_err[[ALLOY_COL, TARGET_COL, 'pred_avg', 'abs_residual']]
            .sort_values('abs_residual', ascending=False).head(10))
    large_err.to_excel('ann_large_error_samples.xlsx', index=False)
    print('Saved -> ann_large_error_samples.xlsx')


In [ ]:
print('=' * 65)
print('SUMMARY REPORT')
print('=' * 65)

report = (
    f'ann Ensemble Prediction Report\n'
    f'=====================================\n'
    f'Generated : {pd.Timestamp.now().stanntime("%Y-%m-%d %H:%M:%S")}\n'
    f'\n'
    f'Param-Set Comparison (all {len(ALL_PARAM_SETS)} sets)\n'
    f'{cmp_df.to_string()}\n'
    f'\n'
    f'Best Param Set (auto-detected, lowest MAE) : Set {BEST_PARAM_SET}\n'
    f'\n'
    f'Model Configuration\n'
    f'  Model directory  : {MODEL_DIR}\n'
    f'  Param set        : {BEST_PARAM_SET}\n'
    f'  Ensemble size    : {ensemble_size}\n'
    f'  Models           : {chr(10).join("    " + m for m in model_list)}\n'
    f'\n'
    f'Data\n'
    f'  Feature file     : {FEAT_FILE or RAW_FEAT_FILE}\n'
    f'  Samples          : {len(df_results)}\n'
    f'  Features         : {FEAT_COLS}\n'
    f'\n'
    f'Peannormance Metrics (Best Set {BEST_PARAM_SET})\n'
    f'  MAE              : {mae:.6f}\n'
    f'  RMSE             : {rmse:.6f}\n'
    f'  MAPE             : {mape:.3f} %\n'
    f'  R2               : {r2:.6f}\n'
    f'\n'
    f'Prediction Statistics\n'
    f'  Mean prediction  : {y_avg.mean():.4f}\n'
    f'  Std  prediction  : {y_avg.std():.4f}\n'
    f'  Min  prediction  : {y_avg.min():.4f}\n'
    f'  Max  prediction  : {y_avg.max():.4f}\n'
    f'\n'
    f'Ensemble Uncertainty\n'
    f'  Mean std         : {ensemble_std.mean():.4f}\n'
    f'  Max  std         : {ensemble_std.max():.4f}\n'
    f'\n'
    f'Residuals\n'
    f'  Mean             : {residuals.mean():.6f}\n'
    f'  Std              : {residuals.std():.6f}\n'
    f'  Min              : {residuals.min():.6f}\n'
    f'  Max              : {residuals.max():.6f}\n'
    f'\n'
    f'Output Files\n'
    f'  ann_all_sets_comparison.csv\n'
    f'  ann_all_sets_bar.png\n'
    f'  ann_all_sets_parity_overlay.png\n'
    f'  ann_all_sets_distributions.png\n'
    f'  {OUTPUT_FILE}\n'
    f'  ann_parity.png\n'
    f'  ann_dashboard.png\n'
    f'  ann_individual_models.png\n'
    f'  ann_high_uncertainty_samples.xlsx  (if flagged)\n'
    f'  ann_large_error_samples.xlsx       (if flagged)\n'
    f'  ann_summary_report.txt\n'
)

print(report)
with open('ann_summary_report.txt', 'w') as fh:
    fh.write(report)
print('Saved -> ann_summary_report.txt')
print('\n' + '=' * 65)
print('COMPLETE')
print('=' * 65)
